In [13]:
import pandas as pd
from sqlalchemy import create_engine, text
from db_rh_connect import engine

In [14]:
employes    = pd.read_csv("data/extrait_sirh.csv")
evaluations = pd.read_csv("data/extrait_eval.csv")
sondage     = pd.read_csv("data/extrait_sondage.csv")

In [15]:
evaluations["id_employee"] = (
    evaluations["eval_number"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype("Int64")
)
sondage["code_sondage"] = sondage["code_sondage"].astype("Int64")

In [16]:
engine = create_engine(
    "postgresql://valerianbarreau@localhost:5432/rh_database"
)

employes.to_sql("employes", engine, if_exists="replace", index=False)
evaluations.to_sql("evaluations", engine, if_exists="replace", index=False)
sondage.to_sql("sondage", engine, if_exists="replace", index=False)

print("✅ Tables importées !")

InternalError: (psycopg2.errors.DependentObjectsStillExist) cannot drop table employes because other objects depend on it
DETAIL:  constraint fk_eval_employe on table evaluations depends on table employes
constraint fk_sondage_employe on table sondage depends on table employes
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE employes]
(Background on this error at: https://sqlalche.me/e/20/2j85)

In [17]:
with engine.connect() as conn:
    conn.execute(text("""
        ALTER TABLE employes    ADD PRIMARY KEY (id_employee);
        ALTER TABLE evaluations ADD PRIMARY KEY (eval_number);
        ALTER TABLE sondage     ADD PRIMARY KEY (code_sondage);

        ALTER TABLE evaluations
            ADD CONSTRAINT fk_eval_employe
            FOREIGN KEY (id_employee) REFERENCES employes(id_employee);

        ALTER TABLE sondage
            ADD CONSTRAINT fk_sondage_employe
            FOREIGN KEY (code_sondage) REFERENCES employes(id_employee);
    """))
    conn.commit()
print("✅ Relations créées !")

ProgrammingError: (psycopg2.errors.InvalidTableDefinition) multiple primary keys for table "employes" are not allowed

[SQL: 
        ALTER TABLE employes    ADD PRIMARY KEY (id_employee);
        ALTER TABLE evaluations ADD PRIMARY KEY (eval_number);
        ALTER TABLE sondage     ADD PRIMARY KEY (code_sondage);

        ALTER TABLE evaluations
            ADD CONSTRAINT fk_eval_employe
            FOREIGN KEY (id_employee) REFERENCES employes(id_employee);

        ALTER TABLE sondage
            ADD CONSTRAINT fk_sondage_employe
            FOREIGN KEY (code_sondage) REFERENCES employes(id_employee);
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)